# Phân loại tin tức VietNamNet bằng Semantic KNN

Notebook này gồm **3 hướng embedding retrieval** theo đúng thứ tự thử nghiệm:

1. **Phần chính**: `AITeamVN/Vietnamese_Embedding`
2. **Appendix A**: `BAAI/bge-m3`
3. **Appendix B**: `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`

Thiết kế này giúp bạn có thể:
- chạy phần `Vietnamese_Embedding` như pipeline chính
- benchmark `bge-m3` trước ở cuối notebook
- benchmark `mpnet multilingual` sau cùng
- xóa nguyên từng appendix mà không ảnh hưởng phần trên

| Khối | Nội dung |
|------|----------|
| **Section 0-9** | Pipeline chính với `Vietnamese_Embedding` |
| **Section 10-13** | Appendix A: `BAAI/bge-m3` |
| **Section 14-17** | Appendix B: `paraphrase-multilingual-mpnet-base-v2` |

> **Biểu diễn văn bản cho cả 3 hướng**: `title + content`
>
> **Notebook này ưu tiên chạy bằng GPU** để giảm thời gian encode embedding.

---
## Section 0 - Setup

Chạy **mỗi lần** mở notebook.

In [ ]:
import importlib
import os
import re
import json
import time
import pickle
import random
import warnings
from contextlib import contextmanager

_REQUIRED = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'torch': 'torch',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'tqdm': 'tqdm',
    'sklearn': 'scikit-learn',
    'transformers': 'transformers',
    'pyarrow': 'pyarrow',
    'pyvi': 'pyvi',
}
_missing = {pkg for mod, pkg in _REQUIRED.items() if importlib.util.find_spec(mod) is None}
if _missing:
    raise SystemExit(f'Thiếu thư viện: {sorted(_missing)}')

warnings.filterwarnings('ignore')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from pyvi import ViTokenizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import AutoTokenizer, AutoModel

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
sns.set_theme(style='whitegrid')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    raise SystemExit('Notebook này đang được thiết kế ưu tiên GPU. Hiện không phát hiện CUDA/GPU.')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

@contextmanager
def timer(name):
    start = time.time()
    yield
    print(f'[TIME] {name}: {time.time() - start:.2f}s')

def log(msg, tag='INFO'):
    print(f'[{tag}] {msg}')

print('Python :', os.sys.version.split()[0])
print('Device :', device)
print('GPU    :', torch.cuda.get_device_name(0))

In [ ]:
NOTEBOOK_DIR = os.getcwd()
DATASET_FOLDER = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'Dataset'))
TEMP_DIR = os.path.join(NOTEBOOK_DIR, 'temp')
RESULTS_DIR = os.path.join(NOTEBOOK_DIR, 'results')
MODEL_DIR = os.path.join(NOTEBOOK_DIR, 'model')
for _d in [TEMP_DIR, RESULTS_DIR, MODEL_DIR]:
    os.makedirs(_d, exist_ok=True)

PROCESSED_DATA_PATH = os.path.join(TEMP_DIR, 'processed_data.pkl')

VI_MODEL_NAME = 'AITeamVN/Vietnamese_Embedding'
VI_MAX_LENGTH = 256
VI_BATCH_SIZE = 8
VI_TOP_K = 7
VI_VOTE_POWER = 2.0
VI_MIN_SIMILARITY = 0.05
VI_TRAIN_EMB_PATH = os.path.join(TEMP_DIR, 'vi_train_embeddings.npz')
VI_TEST_EMB_PATH = os.path.join(TEMP_DIR, 'vi_test_embeddings.npz')
VI_RETRIEVAL_CACHE_PATH = os.path.join(MODEL_DIR, 'vi_retrieval_artifacts.pkl')
VI_LABEL_CONFIG_PATH = os.path.join(MODEL_DIR, 'vi_label_config.json')

BGE_MODEL_NAME = 'BAAI/bge-m3'
BGE_MAX_LENGTH = 256
BGE_BATCH_SIZE = 8
BGE_TOP_K = 7
BGE_VOTE_POWER = 2.0
BGE_MIN_SIMILARITY = 0.05
BGE_TRAIN_EMB_PATH = os.path.join(TEMP_DIR, 'bge_train_embeddings.npz')
BGE_TEST_EMB_PATH = os.path.join(TEMP_DIR, 'bge_test_embeddings.npz')
BGE_RETRIEVAL_CACHE_PATH = os.path.join(MODEL_DIR, 'bge_retrieval_artifacts.pkl')
BGE_LABEL_CONFIG_PATH = os.path.join(MODEL_DIR, 'bge_label_config.json')

MPNET_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
MPNET_MAX_LENGTH = 256
MPNET_BATCH_SIZE = 16
MPNET_TOP_K = 7
MPNET_VOTE_POWER = 2.0
MPNET_MIN_SIMILARITY = 0.05
MPNET_TRAIN_EMB_PATH = os.path.join(TEMP_DIR, 'mpnet_train_embeddings.npz')
MPNET_TEST_EMB_PATH = os.path.join(TEMP_DIR, 'mpnet_test_embeddings.npz')
MPNET_RETRIEVAL_CACHE_PATH = os.path.join(MODEL_DIR, 'mpnet_retrieval_artifacts.pkl')
MPNET_LABEL_CONFIG_PATH = os.path.join(MODEL_DIR, 'mpnet_label_config.json')

TEST_SIZE = 0.15
RANDOM_STATE = 42
TRAIN_SAMPLES_PER_CLASS = 3000

LABEL_MAP = {
    'ban-doc': 'Bạn đọc',
    'bao-ve-nguoi-tieu-dung': 'Bảo vệ người tiêu dùng',
    'bat-dong-san': 'Bất động sản',
    'chinh-tri': 'Chính trị',
    'cong-nghe': 'Công nghệ',
    'dan-toc-ton-giao': 'Dân tộc - Tôn giáo',
    'doi-song': 'Đời sống',
    'du-lich': 'Du lịch',
    'giao-duc': 'Giáo dục',
    'kinh-doanh': 'Kinh doanh',
    'oto-xe-may': 'Ô tô - Xe máy',
    'phap-luat': 'Pháp luật',
    'suc-khoe': 'Sức khỏe',
    'the-gioi': 'Thế giới',
    'the-thao': 'Thể thao',
    'thi-truong-tieu-dung': 'Thị trường tiêu dùng',
    'thoi-su': 'Thời sự',
    'tuan-viet-nam': 'Tuần Việt Nam',
    'van-hoa-giai-tri': 'Văn hóa - Giải trí',
}

print('Dataset folder        :', DATASET_FOLDER)
print('Vietnamese_Embedding  :', VI_MODEL_NAME)
print('Appendix A - bge-m3   :', BGE_MODEL_NAME)
print('Appendix B - mpnet    :', MPNET_MODEL_NAME)

---
## Section 1 - Load Dữ Liệu Thô

Đọc toàn bộ **19 file parquet** từ thư mục Dataset.

In [ ]:
if not os.path.isdir(DATASET_FOLDER):
    raise SystemExit(f'Không tìm thấy Dataset: {DATASET_FOLDER}')
expected = set(LABEL_MAP)
found = {f.replace('.parquet', '') for f in os.listdir(DATASET_FOLDER) if f.endswith('.parquet')}
missing = sorted(expected - found)
if missing:
    raise SystemExit(f'Thiếu parquet: {missing}')
for slug in sorted(expected):
    cols = set(pq.read_schema(os.path.join(DATASET_FOLDER, f'{slug}.parquet')).names)
    if not {'title', 'content'}.issubset(cols):
        raise SystemExit(f'{slug}.parquet thiếu cột title/content: {sorted(cols)}')

records = []
load_rows = []
for fname in sorted(os.listdir(DATASET_FOLDER)):
    if not fname.endswith('.parquet'):
        continue
    slug = fname.replace('.parquet', '')
    if slug not in LABEL_MAP:
        continue
    dfc = pd.read_parquet(os.path.join(DATASET_FOLDER, fname)).copy()
    title_missing = int(dfc['title'].isna().sum())
    content_missing = int(dfc['content'].isna().sum())
    title_empty = int(dfc['title'].fillna('').astype(str).str.strip().eq('').sum())
    content_empty = int(dfc['content'].fillna('').astype(str).str.strip().eq('').sum())
    both_empty = int((dfc['title'].fillna('').astype(str).str.strip().eq('') & dfc['content'].fillna('').astype(str).str.strip().eq('')).sum())
    load_rows.append({
        'file': fname,
        'label': LABEL_MAP[slug],
        'rows': int(len(dfc)),
        'title_missing': title_missing,
        'content_missing': content_missing,
        'title_empty': title_empty,
        'content_empty': content_empty,
        'both_empty': both_empty,
    })
    dfc['label'] = LABEL_MAP[slug]
    dfc['source_file'] = fname
    records.append(dfc[['title', 'content', 'label', 'source_file']])

load_df = pd.DataFrame(load_rows).sort_values('rows', ascending=False)
df_raw = pd.concat(records, ignore_index=True)
raw_total = int(len(df_raw))
df_raw['title'] = df_raw['title'].fillna('').astype(str).str.strip()
df_raw['content'] = df_raw['content'].fillna('').astype(str).str.strip()
removed_both_empty = int(((df_raw['title'] == '') & (df_raw['content'] == '')).sum())
df_raw = df_raw[(df_raw['title'] != '') | (df_raw['content'] != '')].copy()
df_raw['full_text'] = (df_raw['title'] + ' ' + df_raw['content']).str.strip()
df_raw['text_len'] = df_raw['full_text'].str.split().str.len()
df_raw = df_raw.reset_index(drop=True)
label_counts = df_raw['label'].value_counts().sort_values(ascending=False)

print('=' * 80)
print('TỔNG QUAN LOAD DỮ LIỆU')
print('=' * 80)
print('Tổng số dòng thô        :', f'{raw_total:,}')
print('Số dòng bị loại (rỗng)  :', f'{removed_both_empty:,}')
print('Số mẫu sau khi lọc      :', f'{len(df_raw):,}')
print('Số chủ đề               :', df_raw['label'].nunique())
print('Thiếu title (NaN)       :', f"{int(load_df['title_missing'].sum()):,}")
print('Thiếu content (NaN)     :', f"{int(load_df['content_missing'].sum()):,}")
print('Title rỗng              :', f"{int(load_df['title_empty'].sum()):,}")
print('Content rỗng            :', f"{int(load_df['content_empty'].sum()):,}")
print('Cả title+content rỗng   :', f"{int(load_df['both_empty'].sum()):,}")
print()
print('SỐ LƯỢNG THEO CHỦ ĐỀ')
print(label_counts.to_string())
print()
print('CHI TIẾT THEO FILE / CHỦ ĐỀ')
print(load_df.to_string(index=False))


---
## Section 2 - Khám Phá Dữ Liệu (EDA)

Phân tích phân bố class và độ dài văn bản trước khi encode embedding.

In [ ]:
vc = df_raw['label'].value_counts().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(20, 6))
axes[0].bar(vc.index, vc.values, color=['#d95f02' if v < vc.median() * 0.75 else '#1b9e77' for v in vc.values])
axes[0].set_title('Phân bố số bài theo chủ đề')
axes[0].tick_params(axis='x', rotation=65)
axes[0].set_ylabel('Số bài')
sns.histplot(df_raw['text_len'], bins=80, kde=True, color='#457b9d', ax=axes[1])
axes[1].set_title('Độ dài full_text')
axes[1].set_xlabel('Số từ')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '01_eda_overview.png'), bbox_inches='tight')
plt.show()

---
## Section 3 - Tiền Xử Lý Văn Bản

Dùng `title + content`, làm sạch nhẹ, **không loại stopwords** để giữ ngữ nghĩa cho embedding retrieval.

In [ ]:
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^\w\sÀ-ỹ]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return ''
    text = ViTokenizer.tokenize(text)
    return re.sub(r'\s+', ' ', text).strip()

if os.path.exists(PROCESSED_DATA_PATH):
    print(f'[OK] Cache tồn tại: {PROCESSED_DATA_PATH}')
else:
    tqdm.pandas()
    df_proc = df_raw[['title', 'content', 'full_text', 'label', 'source_file']].copy()
    with timer('Tiền xử lý full_text'):
        df_proc['clean_text'] = df_proc['full_text'].progress_apply(normalize_text)
    df_proc = df_proc[df_proc['clean_text'] != ''].reset_index(drop=True)
    classes = sorted(df_proc['label'].unique())
    label2id = {label: idx for idx, label in enumerate(classes)}
    id2label = {idx: label for label, idx in label2id.items()}
    with open(PROCESSED_DATA_PATH, 'wb') as f:
        pickle.dump({'df': df_proc, 'label2id': label2id, 'id2label': id2label}, f)
with open(PROCESSED_DATA_PATH, 'rb') as f:
    cache = pickle.load(f)
df = cache['df']
label2id = cache['label2id']
id2label = cache['id2label']
classes = [id2label[i] for i in range(len(id2label))]
N_CLASSES = len(classes)


---
## Section 4 - Chia Train/Test

Chia dữ liệu theo `stratify` để cả 3 hướng embedding dùng chung một split công bằng.

In [ ]:
_train_parts = []
_test_parts = []
for _label, _grp in df.groupby('label', sort=True):
    _grp = _grp.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
    _n_train = min(TRAIN_SAMPLES_PER_CLASS, len(_grp))
    _train_parts.append(_grp.iloc[:_n_train].copy())
    _test_parts.append(_grp.iloc[_n_train:].copy())

train_meta = pd.concat(_train_parts, ignore_index=True)[['clean_text', 'label']].rename(columns={'clean_text': 'text'})
test_meta = pd.concat(_test_parts, ignore_index=True)[['clean_text', 'label']].rename(columns={'clean_text': 'text'})
train_label_ids = train_meta['label'].map(label2id).to_numpy()
test_label_ids = test_meta['label'].map(label2id).to_numpy()

_split_summary = pd.DataFrame({
    'train': train_meta['label'].value_counts().reindex(classes, fill_value=0),
    'test': test_meta['label'].value_counts().reindex(classes, fill_value=0),
})
_split_summary['total'] = _split_summary['train'] + _split_summary['test']

print('Train:', f'{len(train_meta):,}', '| Test:', f'{len(test_meta):,}')
print('Train samples per class (target):', TRAIN_SAMPLES_PER_CLASS)
print(_split_summary.to_string())

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def encode_texts(texts, model_name, max_length, batch_size):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()
    vectors = []
    num_batches = len(range(0, len(texts), batch_size))
    print(f'[ENCODE] model={model_name} | num_texts={len(texts):,} | batch_size={batch_size} | max_length={max_length} | num_batches={num_batches:,}')
    for start in tqdm(range(0, len(texts), batch_size), desc=f'Encoding {model_name}'):
        batch = texts[start:start + batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            outputs = model(**encoded)
            pooled = outputs.pooler_output if getattr(outputs, 'pooler_output', None) is not None else mean_pool(outputs.last_hidden_state, encoded['attention_mask'])
            pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        vectors.append(pooled.cpu().numpy().astype(np.float32))
    del model
    torch.cuda.empty_cache()
    return np.vstack(vectors)

def run_semantic_knn(train_embeddings, test_embeddings, train_label_ids, test_label_ids, top_k, vote_power, min_similarity):
    index = NearestNeighbors(n_neighbors=min(top_k, len(train_embeddings)), metric='cosine', algorithm='brute')
    index.fit(train_embeddings)
    topk_distances, topk_indices = index.kneighbors(test_embeddings, return_distance=True)
    topk_similarities = np.clip(1.0 - topk_distances, 0.0, 1.0)
    topk_weights = np.where(topk_similarities >= min_similarity, np.power(np.clip(topk_similarities, 1e-6, 1.0), vote_power), 0.0)
    raw_scores = np.zeros((len(test_embeddings), N_CLASSES), dtype=np.float32)
    for row_idx in range(len(test_embeddings)):
        for nbr_pos, train_idx in enumerate(topk_indices[row_idx]):
            raw_scores[row_idx, train_label_ids[train_idx]] += topk_weights[row_idx, nbr_pos]
    y_pred_ids = raw_scores.argmax(axis=1)
    return {'topk_indices': topk_indices, 'topk_similarities': topk_similarities, 'raw_scores': raw_scores, 'y_true_ids': test_label_ids, 'y_pred_ids': y_pred_ids, 'acc': accuracy_score(test_label_ids, y_pred_ids), 'f1w': f1_score(test_label_ids, y_pred_ids, average='weighted'), 'f1m': f1_score(test_label_ids, y_pred_ids, average='macro')}

---
## Section 5 - Vietnamese_Embedding: Encode Embedding

In [ ]:
if os.path.exists(VI_TRAIN_EMB_PATH) and os.path.exists(VI_TEST_EMB_PATH):
    log('Đã có cache Vietnamese_Embedding train/test - bỏ qua encode', 'OK')
    vi_train_embeddings = np.load(VI_TRAIN_EMB_PATH, allow_pickle=True)['embeddings']
    vi_test_embeddings = np.load(VI_TEST_EMB_PATH, allow_pickle=True)['embeddings']
else:
    with timer('Encode train embeddings - Vietnamese_Embedding'):
        vi_train_embeddings = encode_texts(train_meta['text'].tolist(), VI_MODEL_NAME, VI_MAX_LENGTH, VI_BATCH_SIZE)
    np.savez_compressed(VI_TRAIN_EMB_PATH, embeddings=vi_train_embeddings)

    with timer('Encode test embeddings - Vietnamese_Embedding'):
        vi_test_embeddings = encode_texts(test_meta['text'].tolist(), VI_MODEL_NAME, VI_MAX_LENGTH, VI_BATCH_SIZE)  
    np.savez_compressed(VI_TEST_EMB_PATH, embeddings=vi_test_embeddings)
    
print('vi_train_embeddings:', vi_train_embeddings.shape)
print('vi_test_embeddings :', vi_test_embeddings.shape)

---
## Section 6 - Vietnamese_Embedding: Semantic KNN

In [ ]:
with timer('Run Semantic KNN - Vietnamese_Embedding'):
    vi_results = run_semantic_knn(vi_train_embeddings, vi_test_embeddings, train_label_ids, test_label_ids, VI_TOP_K, VI_VOTE_POWER, VI_MIN_SIMILARITY)
print(f"Accuracy   : {vi_results['acc']:.4f}")
print(f"F1-weighted: {vi_results['f1w']:.4f}")
print(f"F1-macro   : {vi_results['f1m']:.4f}")

---
## Section 7 - Vietnamese_Embedding: Đánh Giá

In [ ]:
vi_report_str = classification_report(vi_results['y_true_ids'], vi_results['y_pred_ids'], target_names=classes, digits=4)
with open(os.path.join(RESULTS_DIR, 'vi_classification_report.txt'), 'w', encoding='utf-8') as f: f.write(vi_report_str)
print(vi_report_str)
vi_cm = confusion_matrix(vi_results['y_true_ids'], vi_results['y_pred_ids'], labels=list(range(N_CLASSES)))
vi_report_dict = classification_report(vi_results['y_true_ids'], vi_results['y_pred_ids'], target_names=classes, output_dict=True, zero_division=0)
vi_diag_df = pd.DataFrame([{'class': cls, 'precision': vi_report_dict[cls]['precision'], 'recall': vi_report_dict[cls]['recall'], 'f1': vi_report_dict[cls]['f1-score'], 'support': int(vi_report_dict[cls]['support'])} for cls in classes]).sort_values('f1')
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
sns.heatmap(vi_cm, cmap='Blues', ax=axes[0, 0], xticklabels=classes, yticklabels=classes)
axes[0, 0].set_title('Vietnamese_Embedding - Confusion Matrix'); axes[0, 0].tick_params(axis='x', rotation=90); axes[0, 0].tick_params(axis='y', rotation=0)
sns.barplot(data=vi_diag_df, y='class', x='f1', color='#4c78a8', ax=axes[0, 1]); axes[0, 1].set_xlim(0, 1); axes[0, 1].set_title('Vietnamese_Embedding - F1 theo lớp')
sns.scatterplot(data=vi_diag_df, x='support', y='f1', s=90, color='#f58518', ax=axes[1, 0]); axes[1, 0].set_title('Vietnamese_Embedding - Support vs F1')
sns.histplot(vi_results['topk_similarities'][:, 0], bins=40, kde=True, color='#2a9d8f', ax=axes[1, 1]); axes[1, 1].set_title('Vietnamese_Embedding - Top-1 similarity')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, '02_vi_semantic_knn_eval.png'), bbox_inches='tight'); plt.show()

---
## Section 8 - Vietnamese_Embedding: Phân Tích Lỗi

In [ ]:
vi_error_indices = np.where(vi_results['y_true_ids'] != vi_results['y_pred_ids'])[0]
print('Số mẫu dự đoán sai:', len(vi_error_indices))
vi_error_rows = []
for idx in vi_error_indices[:10]:
    vi_error_rows.append({'sample_idx': int(idx), 'true': id2label[vi_results['y_true_ids'][idx]], 'pred': id2label[vi_results['y_pred_ids'][idx]], 'top1_sim': round(float(vi_results['topk_similarities'][idx, 0]), 4), 'neighbor_labels': [train_meta.iloc[n_idx]['label'] for n_idx in vi_results['topk_indices'][idx]]})
print(pd.DataFrame(vi_error_rows).to_string(index=False, max_colwidth=120))

---
## Section 9 - Vietnamese_Embedding: Export

In [ ]:
vi_label_config = {'embedding_model_name': VI_MODEL_NAME, 'strategy': 'semantic_knn', 'text_representation': 'title + content', 'max_length': VI_MAX_LENGTH, 'n_classes': N_CLASSES, 'classes': classes, 'label2id': label2id, 'id2label': {str(k): v for k, v in id2label.items()}, 'top_k': VI_TOP_K, 'vote_power': VI_VOTE_POWER, 'min_similarity': VI_MIN_SIMILARITY, 'metric': 'cosine'}
with open(VI_LABEL_CONFIG_PATH, 'w', encoding='utf-8') as f: json.dump(vi_label_config, f, ensure_ascii=False, indent=2)
with open(VI_RETRIEVAL_CACHE_PATH, 'wb') as f: pickle.dump({'embedding_model_name': VI_MODEL_NAME, 'metric': 'cosine', 'top_k': VI_TOP_K, 'vote_power': VI_VOTE_POWER, 'min_similarity': VI_MIN_SIMILARITY}, f)
print('Đã lưu:', VI_LABEL_CONFIG_PATH)
print('Đã lưu:', VI_RETRIEVAL_CACHE_PATH)

---
## Section 10 - Appendix A: bge-m3 Encode Embedding

Đây là **appendix thử trước** sau `Vietnamese_Embedding`. Bạn có thể xóa toàn bộ từ section này đến Section 13 mà không ảnh hưởng phần trên.

In [ ]:
if os.path.exists(BGE_TRAIN_EMB_PATH) and os.path.exists(BGE_TEST_EMB_PATH):
    log('Đã có cache bge-m3 train/test - bỏ qua encode', 'OK')
    bge_train_embeddings = np.load(BGE_TRAIN_EMB_PATH, allow_pickle=True)['embeddings']
    bge_test_embeddings = np.load(BGE_TEST_EMB_PATH, allow_pickle=True)['embeddings']
else:
    with timer('Encode train embeddings - bge-m3'):
        bge_train_embeddings = encode_texts(train_meta['text'].tolist(), BGE_MODEL_NAME, BGE_MAX_LENGTH, BGE_BATCH_SIZE)
    with timer('Encode test embeddings - bge-m3'):
        bge_test_embeddings = encode_texts(test_meta['text'].tolist(), BGE_MODEL_NAME, BGE_MAX_LENGTH, BGE_BATCH_SIZE)
    np.savez_compressed(BGE_TRAIN_EMB_PATH, embeddings=bge_train_embeddings)
    np.savez_compressed(BGE_TEST_EMB_PATH, embeddings=bge_test_embeddings)
print('bge_train_embeddings:', bge_train_embeddings.shape)
print('bge_test_embeddings :', bge_test_embeddings.shape)

---
## Section 11 - Appendix A: bge-m3 Semantic KNN

In [ ]:
with timer('Run Semantic KNN - bge-m3'):
    bge_results = run_semantic_knn(bge_train_embeddings, bge_test_embeddings, train_label_ids, test_label_ids, BGE_TOP_K, BGE_VOTE_POWER, BGE_MIN_SIMILARITY)
print(f"Accuracy   : {bge_results['acc']:.4f}")
print(f"F1-weighted: {bge_results['f1w']:.4f}")
print(f"F1-macro   : {bge_results['f1m']:.4f}")

---
## Section 12 - Appendix A: bge-m3 Đánh Giá

In [ ]:
bge_report_str = classification_report(bge_results['y_true_ids'], bge_results['y_pred_ids'], target_names=classes, digits=4)
with open(os.path.join(RESULTS_DIR, 'bge_classification_report.txt'), 'w', encoding='utf-8') as f: f.write(bge_report_str)
print(bge_report_str)
compare_df = pd.DataFrame([{'model': 'Vietnamese_Embedding', 'accuracy': vi_results['acc'], 'f1_weighted': vi_results['f1w'], 'f1_macro': vi_results['f1m']}, {'model': 'BAAI/bge-m3', 'accuracy': bge_results['acc'], 'f1_weighted': bge_results['f1w'], 'f1_macro': bge_results['f1m']}])
print(compare_df.to_string(index=False))

---
## Section 13 - Appendix A: bge-m3 Export

In [ ]:
bge_label_config = {'embedding_model_name': BGE_MODEL_NAME, 'strategy': 'semantic_knn', 'text_representation': 'title + content', 'max_length': BGE_MAX_LENGTH, 'n_classes': N_CLASSES, 'classes': classes, 'label2id': label2id, 'id2label': {str(k): v for k, v in id2label.items()}, 'top_k': BGE_TOP_K, 'vote_power': BGE_VOTE_POWER, 'min_similarity': BGE_MIN_SIMILARITY, 'metric': 'cosine'}
with open(BGE_LABEL_CONFIG_PATH, 'w', encoding='utf-8') as f: json.dump(bge_label_config, f, ensure_ascii=False, indent=2)
with open(BGE_RETRIEVAL_CACHE_PATH, 'wb') as f: pickle.dump({'embedding_model_name': BGE_MODEL_NAME, 'metric': 'cosine', 'top_k': BGE_TOP_K, 'vote_power': BGE_VOTE_POWER, 'min_similarity': BGE_MIN_SIMILARITY}, f)
print('Đã lưu:', BGE_LABEL_CONFIG_PATH)
print('Đã lưu:', BGE_RETRIEVAL_CACHE_PATH)

---
## Section 14 - Appendix B: mpnet Encode Embedding

Đây là appendix thử sau `bge-m3`. Bạn có thể xóa toàn bộ từ section này đến hết notebook mà không ảnh hưởng các phần trước.

In [ ]:
if os.path.exists(MPNET_TRAIN_EMB_PATH) and os.path.exists(MPNET_TEST_EMB_PATH):
    log('Đã có cache mpnet train/test - bỏ qua encode', 'OK')
    mpnet_train_embeddings = np.load(MPNET_TRAIN_EMB_PATH, allow_pickle=True)['embeddings']
    mpnet_test_embeddings = np.load(MPNET_TEST_EMB_PATH, allow_pickle=True)['embeddings']
else:
    with timer('Encode train embeddings - mpnet multilingual'):
        mpnet_train_embeddings = encode_texts(train_meta['text'].tolist(), MPNET_MODEL_NAME, MPNET_MAX_LENGTH, MPNET_BATCH_SIZE)
    with timer('Encode test embeddings - mpnet multilingual'):
        mpnet_test_embeddings = encode_texts(test_meta['text'].tolist(), MPNET_MODEL_NAME, MPNET_MAX_LENGTH, MPNET_BATCH_SIZE)
    np.savez_compressed(MPNET_TRAIN_EMB_PATH, embeddings=mpnet_train_embeddings)
    np.savez_compressed(MPNET_TEST_EMB_PATH, embeddings=mpnet_test_embeddings)
print('mpnet_train_embeddings:', mpnet_train_embeddings.shape)
print('mpnet_test_embeddings :', mpnet_test_embeddings.shape)

---
## Section 15 - Appendix B: mpnet Semantic KNN

In [ ]:
with timer('Run Semantic KNN - mpnet multilingual'):
    mpnet_results = run_semantic_knn(mpnet_train_embeddings, mpnet_test_embeddings, train_label_ids, test_label_ids, MPNET_TOP_K, MPNET_VOTE_POWER, MPNET_MIN_SIMILARITY)
print(f"Accuracy   : {mpnet_results['acc']:.4f}")
print(f"F1-weighted: {mpnet_results['f1w']:.4f}")
print(f"F1-macro   : {mpnet_results['f1m']:.4f}")

---
## Section 16 - Appendix B: mpnet Đánh Giá

In [ ]:
mpnet_report_str = classification_report(mpnet_results['y_true_ids'], mpnet_results['y_pred_ids'], target_names=classes, digits=4)
with open(os.path.join(RESULTS_DIR, 'mpnet_classification_report.txt'), 'w', encoding='utf-8') as f: f.write(mpnet_report_str)
print(mpnet_report_str)
compare_df = pd.DataFrame([
    {'model': 'Vietnamese_Embedding', 'accuracy': vi_results['acc'], 'f1_weighted': vi_results['f1w'], 'f1_macro': vi_results['f1m']},
    {'model': 'BAAI/bge-m3', 'accuracy': bge_results['acc'], 'f1_weighted': bge_results['f1w'], 'f1_macro': bge_results['f1m']},
    {'model': 'mpnet multilingual', 'accuracy': mpnet_results['acc'], 'f1_weighted': mpnet_results['f1w'], 'f1_macro': mpnet_results['f1m']},
])
print(compare_df.to_string(index=False))

---
## Section 17 - Appendix B: mpnet Export

In [ ]:
mpnet_label_config = {'embedding_model_name': MPNET_MODEL_NAME, 'strategy': 'semantic_knn', 'text_representation': 'title + content', 'max_length': MPNET_MAX_LENGTH, 'n_classes': N_CLASSES, 'classes': classes, 'label2id': label2id, 'id2label': {str(k): v for k, v in id2label.items()}, 'top_k': MPNET_TOP_K, 'vote_power': MPNET_VOTE_POWER, 'min_similarity': MPNET_MIN_SIMILARITY, 'metric': 'cosine'}
with open(MPNET_LABEL_CONFIG_PATH, 'w', encoding='utf-8') as f: json.dump(mpnet_label_config, f, ensure_ascii=False, indent=2)
with open(MPNET_RETRIEVAL_CACHE_PATH, 'wb') as f: pickle.dump({'embedding_model_name': MPNET_MODEL_NAME, 'metric': 'cosine', 'top_k': MPNET_TOP_K, 'vote_power': MPNET_VOTE_POWER, 'min_similarity': MPNET_MIN_SIMILARITY}, f)
print('Đã lưu:', MPNET_LABEL_CONFIG_PATH)
print('Đã lưu:', MPNET_RETRIEVAL_CACHE_PATH)